In [1]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from scipy.stats import kendalltau
from sentence_transformers import SentenceTransformer

# ---- Project path setup (Notebook Safe) ----
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
    
# Import SPARQLCE components now that the path is set
from sparqlce.embedder import SPARQLEmbedder

print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings


In [12]:
# ==========================================
# Define your paths using the absolute root
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Adjust these filenames if your robustness dataset is named differently
ROBUSTNESS_FILE = os.path.join(_PROJECT_ROOT, "datasets", "robustness_pairs.json") 
ROBUSTNESS_FILE = os.path.join(_PROJECT_ROOT, "datasets", "eval_RQ1_dbpedia.json")

TEST_FILE = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia-triplets-dataset.json")
SPARQLCE_CKPT = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")
OUTPUT_DIR = os.path.join(_PROJECT_ROOT, "results", "robustness")

print(f"Using device: {DEVICE}")

Using device: cuda


In [13]:
# ==========================================
# 1. Model wrappers
# ==========================================
class SparqlCEEncoder:
    def __init__(self, ckpt_path, device='cuda'):
        self.device = device
        self.embedder = SPARQLEmbedder(ckpt_path)

    def encode(self, queries, batch_size=32):
        if hasattr(self.embedder, 'embed'):
            return self.embedder.embed(queries)
        if hasattr(self.embedder, 'encode'):
            return self.embedder.encode(queries)
        raise AttributeError("SPARQLEmbedder does not expose embed() or encode()")

class E5Encoder:
    def __init__(self, model_name='intfloat/e5-base-v2', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)
        
    def encode(self, queries):
        prefixed = [f'query: {q}' for q in queries]
        return self.model.encode(prefixed, convert_to_numpy=True, show_progress_bar=False)

class QwenEncoder:
    def __init__(self, model_name='Qwen/Qwen2.5-0.5B', device='cuda'):
        self.model = SentenceTransformer(model_name, device=device)
        
    def encode(self, queries):
        return self.model.encode(queries, convert_to_numpy=True, show_progress_bar=False)

class TFIDFEncoder:
    def __init__(self, corpus):
        self.vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=10000)
        self.embeddings = self.vectorizer.fit_transform(corpus).toarray()
        
    def encode(self, queries):
        return self.vectorizer.transform(queries).toarray()

In [14]:
# ==========================================
# 2. Metrics Functions
# ==========================================
def compute_cosine_sim(emb_orig, emb_pert):
    sim = F.cosine_similarity(torch.tensor(emb_orig), torch.tensor(emb_pert), dim=1)
    return sim.numpy()

def retrieval_rank_metrics(emb_orig, emb_pert, corpus_emb, top_k=10):
    orig_scores = sk_cosine(emb_orig, corpus_emb)
    pert_scores = sk_cosine(emb_pert, corpus_emb)
    n = emb_orig.shape[0]
    jaccard = []
    tau_vals = []
    for i in range(n):
        idx_orig = np.argsort(-orig_scores[i])[:top_k]
        idx_pert = np.argsort(-pert_scores[i])[:top_k]
        inter = len(set(idx_orig) & set(idx_pert))
        jaccard.append(inter / top_k)
        tau, _ = kendalltau(orig_scores[i], pert_scores[i])
        tau_vals.append(0.0 if np.isnan(tau) else tau)
    return np.mean(jaccard), np.mean(tau_vals)

def mrr_retention(anchor_emb_orig, anchor_emb_pert, corpus_emb, pos_indices_list):
    mrr_orig = []
    mrr_pert = []
    for i, pos_idx in enumerate(pos_indices_list):
        if len(pos_idx) == 0:
            continue
        # original MRR
        scores_orig = sk_cosine(anchor_emb_orig[i:i+1], corpus_emb).flatten()
        best_orig = scores_orig[pos_idx].max()
        rank_orig = 1 + np.sum(scores_orig > best_orig)
        mrr_orig.append(1.0 / rank_orig)
        
        # perturbed MRR
        scores_pert = sk_cosine(anchor_emb_pert[i:i+1], corpus_emb).flatten()
        best_pert = scores_pert[pos_idx].max()
        rank_pert = 1 + np.sum(scores_pert > best_pert)
        mrr_pert.append(1.0 / rank_pert)
    return np.mean(mrr_orig) if mrr_orig else 0, np.mean(mrr_pert) if mrr_pert else 0

In [15]:
# ==========================================
# 3. Load Data & Build Corpus
# ==========================================
with open(ROBUSTNESS_FILE) as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs)} robustness pairs")

for i, p in enumerate(pairs):
    p['_idx'] = i

with open(TEST_FILE) as f:
    test_data_loaded = json.load(f)

corpus_queries = []
query_to_idx = {}
anchor_pos_indices = defaultdict(list)

for item in test_data_loaded:
    anchor = item.get('query', item.get('anchor_query', '')).strip()
    if not anchor:
        continue
    if anchor not in query_to_idx:
        query_to_idx[anchor] = len(corpus_queries)
        corpus_queries.append(anchor)

    # positives
    for p in item.get('positive_queries', []):
        p_text = p['query'].strip() if isinstance(p, dict) else p.strip()
        if p_text not in query_to_idx:
            query_to_idx[p_text] = len(corpus_queries)
            corpus_queries.append(p_text)
        anchor_pos_indices[anchor].append(query_to_idx[p_text])

    # negatives
    for n in item.get('negative_queries', []):
        n_text = n['query'].strip() if isinstance(n, dict) else n.strip()
        if n_text not in query_to_idx:
            query_to_idx[n_text] = len(corpus_queries)
            corpus_queries.append(n_text)

print(f"Corpus size: {len(corpus_queries)}")
print(f"Test anchors with positives: {len(anchor_pos_indices)}")

pert_by_anchor = defaultdict(list)
for p in pairs:
    orig = p['original'].strip()
    pert = p['perturbed'].strip()
    p['original'] = orig
    p['perturbed'] = pert
    pert_by_anchor[orig].append(p)

Loaded 7578 robustness pairs
Corpus size: 4727
Test anchors with positives: 4713


In [16]:
# ==========================================
# 4. Instantiate Models & Evaluate
# ==========================================
models = {
    'sparqlce': SparqlCEEncoder(SPARQLCE_CKPT, DEVICE),
    'e5': E5Encoder(device=DEVICE),
    'qwen': QwenEncoder(device=DEVICE),
    'tfidf': TFIDFEncoder(corpus_queries),
}

results = {}

for name, encoder in models.items():
    print(f"\n=== Evaluating {name} ===")
    
    originals = [p['original'] for p in pairs]
    perturbeds = [p['perturbed'] for p in pairs]
    emb_orig = encoder.encode(originals)
    emb_pert = encoder.encode(perturbeds)

    cos_sim = compute_cosine_sim(emb_orig, emb_pert)
    corpus_emb = encoder.encode(corpus_queries)

    mrr_orig_total = []
    mrr_pert_total = []
    
    for anchor, pos_idx in anchor_pos_indices.items():
        if anchor not in pert_by_anchor:
            continue  
            
        anchor_idx = query_to_idx[anchor]
        anchor_emb_orig = corpus_emb[anchor_idx:anchor_idx+1]
        
        scores_orig = sk_cosine(anchor_emb_orig, corpus_emb).flatten()
        best_orig = scores_orig[pos_idx].max()
        rank_orig = 1 + np.sum(scores_orig > best_orig)
        mrr_orig_total.append(1.0 / rank_orig)

        mrr_pert_versions = []
        for pert_pair in pert_by_anchor[anchor]:
            pert_idx = pert_pair['_idx']
            pert_emb = emb_pert[pert_idx:pert_idx+1]
            scores_pert = sk_cosine(pert_emb, corpus_emb).flatten()
            best_pert = scores_pert[pos_idx].max()
            rank_pert = 1 + np.sum(scores_pert > best_pert)
            mrr_pert_versions.append(1.0 / rank_pert)
            
        if mrr_pert_versions:
            mrr_pert_total.append(np.mean(mrr_pert_versions))

    mean_mrr_orig = np.mean(mrr_orig_total) if mrr_orig_total else 0
    mean_mrr_pert = np.mean(mrr_pert_total) if mrr_pert_total else 0
    mrr_drop = mean_mrr_orig - mean_mrr_pert

    by_type = defaultdict(list)
    for p in pairs:
        by_type[p['perturbation_type']].append(p['_idx'])
        
    per_type = {}
    for ptype, idxs in by_type.items():
        type_cos = cos_sim[idxs].mean()
        per_type[ptype] = {'mean_cos_sim': float(type_cos)}

    results[name] = {
        'overall_mean_cos_sim': float(cos_sim.mean()),
        'overall_std_cos_sim': float(cos_sim.std()),
        'mrr_original': float(mean_mrr_orig),
        'mrr_perturbed': float(mean_mrr_pert),
        'mrr_drop': float(mrr_drop),
        'per_type': per_type
    }

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt...


No sentence-transformers model found with name Qwen/Qwen2.5-0.5B. Creating a new one with mean pooling.



=== Evaluating sparqlce ===

=== Evaluating e5 ===

=== Evaluating qwen ===

=== Evaluating tfidf ===


In [17]:
# ==========================================
# 5. Save Results & Print Summary
# ==========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, 'robustness_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print("\n========== Robustness Comparison ==========")
print(f"{'Model':<10} {'CosSim':>8} {'MRR Orig':>8} {'MRR Pert':>8} {'MRR Drop':>8}")
for name in models:
    r = results[name]
    print(f"{name:<10} {r['overall_mean_cos_sim']:8.4f} {r['mrr_original']:8.4f} {r['mrr_perturbed']:8.4f} {r['mrr_drop']:8.4f}")

print("\nPer perturbation type (mean cosine similarity):")
for name in models:
    print(f"\n{name}:")
    for ptype, stats in results[name]['per_type'].items():
        print(f"  {ptype:<25}: {stats['mean_cos_sim']:.4f}")
        
print(f"\n✓ Detailed results saved to: {out_path}")


========== Robustness Comparison ==========
Model        CosSim MRR Orig MRR Pert MRR Drop
sparqlce     0.9910   0.3023   0.2909   0.0114
e5           0.9889   0.1479   0.1422   0.0057
qwen         0.9983   0.1530   0.1433   0.0097
tfidf        0.9700   0.2219   0.2141   0.0078

Per perturbation type (mean cosine similarity):

sparqlce:
  variable_renaming        : 1.0000
  redundant_optional       : 1.0000
  filter_true              : 0.9825
  distinct_toggle          : 0.9767
  triple_shuffle           : 0.9999
  union_duplication        : 1.0000
  subquery_wrapping        : 1.0000

e5:
  variable_renaming        : 0.9884
  redundant_optional       : 0.9911
  filter_true              : 0.9910
  distinct_toggle          : 0.9793
  triple_shuffle           : 0.9989
  union_duplication        : 0.9892
  subquery_wrapping        : 0.9979

qwen:
  variable_renaming        : 0.9993
  redundant_optional       : 0.9960
  filter_true              : 0.9981
  distinct_toggle          : 0.9994
